# 06 Drought Feature Enrichment

This notebook adds **U.S. Drought Monitor (USDM)** drought features to the wildfire modeling table.

Input:

```text
data/processed/calfire_with_gridmet_terrain.csv
```

Outputs:

```text
data/raw/drought/usdm_tiff/USDM_YYYYMMDD.tif
data/interim/usdm_tiff_download_log.csv
data/interim/usdm_missing_weeks.csv
data/processed/drought_features.csv
data/processed/calfire_with_gridmet_terrain_drought.csv
```

Workflow:

```text
fire start date
→ latest USDM week available before the fire date
→ download gridded USDM GeoTIFF for that week
→ sample drought class at the fire latitude/longitude
→ encode drought category/intensity
→ merge drought features into the modeling table
```

This version fixes the earlier sampling bug where `usdm_missing_reason` was initialized as a float column and then failed when assigning text labels.


## 0. Package setup

Install once if needed:

```powershell
pip install pandas numpy geopandas rasterio requests tqdm shapely
```

If Windows has issues with geospatial packages, use conda/mamba:

```powershell
conda install -c conda-forge pandas numpy geopandas rasterio requests tqdm shapely
```


In [1]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import requests
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 180)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"

RAW_DROUGHT_DIR = RAW_DIR / "drought" / "usdm_tiff"

for path in [RAW_DROUGHT_DIR, INTERIM_DIR, PROCESSED_DIR]:
    path.mkdir(parents=True, exist_ok=True)

INPUT_PATH = PROCESSED_DIR / "calfire_with_gridmet_terrain.csv"
DROUGHT_FEATURES_PATH = PROCESSED_DIR / "drought_features.csv"
OUTPUT_PATH = PROCESSED_DIR / "calfire_with_gridmet_terrain_drought.csv"

USDM_TIFF_BASE_URL = "https://www.ncei.noaa.gov/pub/data/nidis/geojson/us/usdm-tiff/wgs84-tiff"

print("Project root:", PROJECT_ROOT)
print("Input path:", INPUT_PATH)
print("Raw drought folder:", RAW_DROUGHT_DIR)
print("Drought features output:", DROUGHT_FEATURES_PATH)
print("Merged output:", OUTPUT_PATH)


Project root: c:\Users\chaud\Desktop\repositories\wildfire-severity-v2
Input path: c:\Users\chaud\Desktop\repositories\wildfire-severity-v2\data\processed\calfire_with_gridmet_terrain.csv
Raw drought folder: c:\Users\chaud\Desktop\repositories\wildfire-severity-v2\data\raw\drought\usdm_tiff
Drought features output: c:\Users\chaud\Desktop\repositories\wildfire-severity-v2\data\processed\drought_features.csv
Merged output: c:\Users\chaud\Desktop\repositories\wildfire-severity-v2\data\processed\calfire_with_gridmet_terrain_drought.csv


c:\Users\chaud\Desktop\repositories\wildfire-severity-v2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load current modeling table

In [2]:
fires = pd.read_csv(INPUT_PATH)

required_cols = ["gridmet_id", "Latitude", "Longitude"]
missing = [col for col in required_cols if col not in fires.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

fires["gridmet_id"] = fires["gridmet_id"].astype(str)
fires["Latitude"] = pd.to_numeric(fires["Latitude"], errors="coerce")
fires["Longitude"] = pd.to_numeric(fires["Longitude"], errors="coerce")

if "fire_start_date" in fires.columns:
    fires["fire_start_date"] = pd.to_datetime(fires["fire_start_date"], errors="coerce")
elif "StartedDateOnly" in fires.columns:
    fires["fire_start_date"] = pd.to_datetime(fires["StartedDateOnly"], errors="coerce")
elif "started_date" in fires.columns:
    fires["fire_start_date"] = pd.to_datetime(fires["started_date"], errors="coerce")
else:
    raise ValueError("Could not find fire_start_date, StartedDateOnly, or started_date.")

pre_drop = len(fires)
fires = fires.dropna(subset=["gridmet_id", "Latitude", "Longitude", "fire_start_date"]).copy()
fires = fires[fires["Latitude"].between(32, 42)].copy()
fires = fires[fires["Longitude"].between(-125, -113)].copy()

print("Rows loaded:", pre_drop)
print("Rows after date/coordinate filter:", len(fires))
display(fires[["gridmet_id", "Name", "fire_start_date", "Latitude", "Longitude", "AcresBurned", "severity_tier"]].head())


Rows loaded: 2397
Rows after date/coordinate filter: 2397


,gridmet_id,Name,fire_start_date,Latitude,Longitude,AcresBurned,severity_tier
0,fire_00000,Creek Fire,2016-10-10,38.409580,-122.431720,65.0,Small
1,fire_00001,Taglio Fire,2016-04-24,37.217100,-121.080360,30.0,Small
2,fire_00002,Tulloch Fire,2016-05-30,37.927613,-120.528836,85.0,Small
3,fire_00003,Metz Fire,2016-05-22,36.381230,-121.200590,3876.0,Large
4,fire_00004,Wheatland Fire,2016-05-23,34.276000,-118.354000,156.0,Medium


## 2. Assign each fire to the latest published USDM week

USDM map dates are Tuesdays, but maps are released on Thursdays. To avoid leakage, this uses the most recent Tuesday map whose Thursday release date was already available on or before the fire start date.


In [3]:
def previous_tuesday(date):
    date = pd.Timestamp(date)
    days_since_tuesday = (date.weekday() - 1) % 7
    return date - pd.Timedelta(days=days_since_tuesday)


def latest_published_usdm_week(date):
    date = pd.Timestamp(date)
    candidate_tuesday = previous_tuesday(date)
    candidate_release = candidate_tuesday + pd.Timedelta(days=2)

    if candidate_release <= date:
        return candidate_tuesday
    return candidate_tuesday - pd.Timedelta(days=7)


fires["usdm_week"] = fires["fire_start_date"].apply(latest_published_usdm_week)
fires["usdm_week_str"] = fires["usdm_week"].dt.strftime("%Y%m%d")

needed_weeks = sorted(fires["usdm_week_str"].unique())

print("Unique USDM weeks needed:", len(needed_weeks))
print("First 10:", needed_weeks[:10])
print("Last 10:", needed_weeks[-10:])
display(fires[["gridmet_id", "fire_start_date", "usdm_week", "usdm_week_str"]].head(10))


Unique USDM weeks needed: 301
First 10: ['20160419', '20160503', '20160510', '20160517', '20160524', '20160531', '20160607', '20160614', '20160621', '20160628']
Last 10: ['20241015', '20241022', '20241029', '20241105', '20241112', '20241119', '20241126', '20241203', '20241210', '20241217']


,gridmet_id,fire_start_date,usdm_week,usdm_week_str
0,fire_00000,2016-10-10,2016-10-04,20161004
1,fire_00001,2016-04-24,2016-04-19,20160419
2,fire_00002,2016-05-30,2016-05-24,20160524
3,fire_00003,2016-05-22,2016-05-17,20160517
4,fire_00004,2016-05-23,2016-05-17,20160517
5,fire_00005,2016-05-25,2016-05-17,20160517
6,fire_00006,2016-06-07,2016-05-31,20160531
7,fire_00007,2016-05-27,2016-05-24,20160524
8,fire_00008,2016-05-29,2016-05-24,20160524
9,fire_00009,2016-05-10,2016-05-03,20160503


## 3. Download gridded USDM GeoTIFFs

Files are downloaded from a predictable NCEI path:

```text
USDM_YYYYMMDD.tif
```

If you already downloaded the files, this cell will skip them.


In [4]:
def download_usdm_tiff(date_str, overwrite=False, timeout=90):
    out_path = RAW_DROUGHT_DIR / f"USDM_{date_str}.tif"

    if out_path.exists() and not overwrite:
        return {
            "usdm_week_str": date_str,
            "status": "exists",
            "path": str(out_path),
            "url": f"{USDM_TIFF_BASE_URL}/USDM_{date_str}.tif",
        }

    url = f"{USDM_TIFF_BASE_URL}/USDM_{date_str}.tif"

    try:
        r = requests.get(url, timeout=timeout)
        r.raise_for_status()
        content = r.content

        looks_like_tiff = content[:2] in [b"II", b"MM"]
        if len(content) < 1000 or not looks_like_tiff:
            return {
                "usdm_week_str": date_str,
                "status": "invalid_or_not_tiff",
                "path": str(out_path),
                "url": url,
            }

        out_path.write_bytes(content)

        return {
            "usdm_week_str": date_str,
            "status": "downloaded",
            "path": str(out_path),
            "url": url,
        }

    except Exception as e:
        return {
            "usdm_week_str": date_str,
            "status": f"error: {e}",
            "path": str(out_path),
            "url": url,
        }


OVERWRITE_DROUGHT = False

download_log = []

for week in tqdm(needed_weeks):
    result = download_usdm_tiff(week, overwrite=OVERWRITE_DROUGHT)
    download_log.append(result)
    time.sleep(0.15)

download_log_df = pd.DataFrame(download_log)
download_log_df.to_csv(INTERIM_DIR / "usdm_tiff_download_log.csv", index=False)

print("Download status counts:")
display(download_log_df["status"].value_counts())
display(download_log_df.head())


100%|██████████| 301/301 [00:45<00:00,  6.56it/s]

Download status counts:


status
exists    301
Name: count, dtype: int64

,usdm_week_str,status,path,url
0,20160419,exists,c:\Users\chaud\Desktop\repositories\wildfire-s...,https://www.ncei.noaa.gov/pub/data/nidis/geojs...
1,20160503,exists,c:\Users\chaud\Desktop\repositories\wildfire-s...,https://www.ncei.noaa.gov/pub/data/nidis/geojs...
2,20160510,exists,c:\Users\chaud\Desktop\repositories\wildfire-s...,https://www.ncei.noaa.gov/pub/data/nidis/geojs...
3,20160517,exists,c:\Users\chaud\Desktop\repositories\wildfire-s...,https://www.ncei.noaa.gov/pub/data/nidis/geojs...
4,20160524,exists,c:\Users\chaud\Desktop\repositories\wildfire-s...,https://www.ncei.noaa.gov/pub/data/nidis/geojs...


## 4. Check missing USDM files

In [5]:
existing_weeks = {
    path.stem.replace("USDM_", "")
    for path in RAW_DROUGHT_DIR.glob("USDM_*.tif")
}

missing_after_download = [week for week in needed_weeks if week not in existing_weeks]

print("Needed weeks:", len(needed_weeks))
print("Existing/downloaded weeks:", len(existing_weeks))
print("Still missing:", len(missing_after_download))
print(missing_after_download[:30])

pd.DataFrame({"usdm_week_str": missing_after_download}).to_csv(
    INTERIM_DIR / "usdm_missing_weeks.csv",
    index=False
)


Needed weeks: 301
Existing/downloaded weeks: 301
Still missing: 0
[]


## 5. Inspect one USDM raster

This confirms the file can be opened and shows its CRS, nodata value, and sampled value range.


In [6]:
sample_raster_path = next(iter(sorted(RAW_DROUGHT_DIR.glob("USDM_*.tif"))), None)

if sample_raster_path is None:
    raise FileNotFoundError("No USDM TIFF files found.")

with rasterio.open(sample_raster_path) as src:
    arr = src.read(1)
    print("Sample raster:", sample_raster_path.name)
    print("CRS:", src.crs)
    print("Shape:", arr.shape)
    print("Nodata:", src.nodata)
    print("Raw min:", np.nanmin(arr))
    print("Raw max:", np.nanmax(arr))
    print("Unique values sample:", np.unique(arr)[:20])


Sample raster: USDM_20160419.tif
CRS: EPSG:4326
Shape: (2142, 14358)
Nodata: -9.0
Raw min: -9
Raw max: 4
Unique values sample: [-9 -1  0  1  2  3  4]


## 6. Convert fires to geospatial points

In [7]:
fires_gdf = gpd.GeoDataFrame(
    fires[["gridmet_id", "Name", "fire_start_date", "usdm_week", "usdm_week_str", "Latitude", "Longitude"]].copy(),
    geometry=gpd.points_from_xy(fires["Longitude"], fires["Latitude"]),
    crs="EPSG:4326",
)

print("Fires GeoDataFrame:", fires_gdf.shape)
display(fires_gdf.head())


Fires GeoDataFrame: (2397, 8)


,gridmet_id,Name,fire_start_date,usdm_week,usdm_week_str,Latitude,Longitude,geometry
0,fire_00000,Creek Fire,2016-10-10,2016-10-04,20161004,38.409580,-122.431720,POINT (-122.43172 38.40958)
1,fire_00001,Taglio Fire,2016-04-24,2016-04-19,20160419,37.217100,-121.080360,POINT (-121.08036 37.2171)
2,fire_00002,Tulloch Fire,2016-05-30,2016-05-24,20160524,37.927613,-120.528836,POINT (-120.52884 37.92761)
3,fire_00003,Metz Fire,2016-05-22,2016-05-17,20160517,36.381230,-121.200590,POINT (-121.20059 36.38123)
4,fire_00004,Wheatland Fire,2016-05-23,2016-05-17,20160517,34.276000,-118.354000,POINT (-118.354 34.276)


## 7. Sample drought values at fire points

This is the corrected sampling function.

The key fix is that `usdm_missing_reason` is created as an object/text column from the start, so assigning text labels does not crash the sampling step.


In [8]:
def sample_usdm_tiff_for_week(fires_week):
    date_str = fires_week["usdm_week_str"].iloc[0]
    raster_path = RAW_DROUGHT_DIR / f"USDM_{date_str}.tif"

    out = fires_week[["gridmet_id", "usdm_week", "usdm_week_str"]].copy()
    out["usdm_raw"] = np.nan
    out["usdm_missing_reason"] = pd.Series([None] * len(out), index=out.index, dtype="object")

    if not raster_path.exists():
        out["usdm_missing_reason"] = "missing_tiff"
        return out

    try:
        with rasterio.open(raster_path) as src:
            points_projected = fires_week.to_crs(src.crs)
            coords = [(geom.x, geom.y) for geom in points_projected.geometry]
            values = [sample[0] for sample in src.sample(coords)]

            out["usdm_raw"] = pd.to_numeric(values, errors="coerce")

            nodata_values = {-9999, -999, 255}
            if src.nodata is not None:
                nodata_values.add(src.nodata)

            nodata_mask = out["usdm_raw"].isin(nodata_values)
            out.loc[nodata_mask, "usdm_missing_reason"] = "no_drought_or_nodata"

    except Exception as e:
        out["usdm_raw"] = np.nan
        out["usdm_missing_reason"] = f"sample_error: {e}"

    return out


parts = []

for week, group in tqdm(fires_gdf.groupby("usdm_week_str")):
    parts.append(sample_usdm_tiff_for_week(group))

drought_features = pd.concat(parts, ignore_index=True)

print("Drought feature rows:", drought_features.shape)
display(drought_features.head())

print("Raw USDM value counts:")
display(drought_features["usdm_raw"].value_counts(dropna=False).sort_index())

print("Missing reason counts:")
display(drought_features["usdm_missing_reason"].value_counts(dropna=False))


100%|██████████| 301/301 [00:05<00:00, 54.80it/s]

Drought feature rows: (2397, 5)


,gridmet_id,usdm_week,usdm_week_str,usdm_raw,usdm_missing_reason
0,fire_00001,2016-04-19,20160419,3,None
1,fire_00009,2016-05-03,20160503,4,None
2,fire_00010,2016-05-10,20160510,4,None
3,fire_00011,2016-05-10,20160510,3,None
4,fire_00012,2016-05-10,20160510,2,None


Raw USDM value counts:


usdm_raw
-9      10
-1    1249
 0     429
 1     207
 2     221
 3     169
 4     112
Name: count, dtype: int64

Missing reason counts:


usdm_missing_reason
None                    2387
no_drought_or_nodata      10
Name: count, dtype: int64

## 8. Encode drought categories

Expected gridded USDM encoding:

```text
-9999 = None / no drought / wet
0     = D0, Abnormally Dry
1     = D1, Moderate Drought
2     = D2, Severe Drought
3     = D3, Extreme Drought
4     = D4, Exceptional Drought
```

For modeling:

```text
None/no drought = 0
D0              = 0
D1              = 1
D2              = 2
D3              = 3
D4              = 4
```

This treats D0 as abnormally dry but not actual drought.


In [10]:
def usdm_category_from_raw(value):
    if pd.isna(value):
        return np.nan

    try:
        value = int(value)
    except Exception:
        return np.nan

    mapping = {
        -9999: "None",
        -999: "None",
        255: "None",
        0: "D0",
        1: "D1",
        2: "D2",
        3: "D3",
        4: "D4",
    }

    return mapping.get(value, np.nan)


def usdm_intensity_from_category(category):
    if pd.isna(category):
        return np.nan

    mapping = {
        "None": 0,
        "D0": 0,
        "D1": 1,
        "D2": 2,
        "D3": 3,
        "D4": 4,
    }

    return mapping.get(category, np.nan)


drought_features["usdm_category"] = drought_features["usdm_raw"].apply(usdm_category_from_raw)
drought_features["usdm_intensity"] = drought_features["usdm_category"].apply(usdm_intensity_from_category)

# Use numeric 0/1 flags instead of bools so missing values are allowed.
valid_intensity = drought_features["usdm_intensity"].notna()

drought_features["is_abnormally_dry_D0_plus"] = np.where(
    valid_intensity,
    drought_features["usdm_category"].isin(["D0", "D1", "D2", "D3", "D4"]).astype(int),
    np.nan
)

drought_features["is_drought_D1_plus"] = np.where(
    valid_intensity,
    (drought_features["usdm_intensity"] >= 1).astype(int),
    np.nan
)

drought_features["is_severe_D2_plus"] = np.where(
    valid_intensity,
    (drought_features["usdm_intensity"] >= 2).astype(int),
    np.nan
)

drought_features["is_extreme_D3_plus"] = np.where(
    valid_intensity,
    (drought_features["usdm_intensity"] >= 3).astype(int),
    np.nan
)

drought_features.to_csv(DROUGHT_FEATURES_PATH, index=False)

print("Saved:", DROUGHT_FEATURES_PATH)
print("Drought features shape:", drought_features.shape)

display(drought_features.head())

print("USDM category counts:")
display(drought_features["usdm_category"].value_counts(dropna=False).sort_index())

print("USDM intensity counts:")
display(drought_features["usdm_intensity"].value_counts(dropna=False).sort_index())

print("Indicator feature means:")
display(drought_features[
    [
        "is_abnormally_dry_D0_plus",
        "is_drought_D1_plus",
        "is_severe_D2_plus",
        "is_extreme_D3_plus",
    ]
].mean().to_frame("fraction"))

Saved: c:\Users\chaud\Desktop\repositories\wildfire-severity-v2\data\processed\drought_features.csv
Drought features shape: (2397, 11)


,gridmet_id,usdm_week,usdm_week_str,usdm_raw,usdm_missing_reason,usdm_category,usdm_intensity,is_abnormally_dry_D0_plus,is_drought_D1_plus,is_severe_D2_plus,is_extreme_D3_plus
0,fire_00001,2016-04-19,20160419,3,None,D3,3.0,1.0,1.0,1.0,1.0
1,fire_00009,2016-05-03,20160503,4,None,D4,4.0,1.0,1.0,1.0,1.0
2,fire_00010,2016-05-10,20160510,4,None,D4,4.0,1.0,1.0,1.0,1.0
3,fire_00011,2016-05-10,20160510,3,None,D3,3.0,1.0,1.0,1.0,1.0
4,fire_00012,2016-05-10,20160510,2,None,D2,2.0,1.0,1.0,1.0,0.0


USDM category counts:


usdm_category
D0      429
D1      207
D2      221
D3      169
D4      112
NaN    1259
Name: count, dtype: int64

USDM intensity counts:


usdm_intensity
0.0     429
1.0     207
2.0     221
3.0     169
4.0     112
NaN    1259
Name: count, dtype: int64

Indicator feature means:


,fraction
is_abnormally_dry_D0_plus,1.000000
is_drought_D1_plus,0.623023
is_severe_D2_plus,0.441125
is_extreme_D3_plus,0.246924


## 9. Merge drought features into the modeling table

In [ ]:
base = pd.read_csv(INPUT_PATH)
base["gridmet_id"] = base["gridmet_id"].astype(str)

drought_features = pd.read_csv(DROUGHT_FEATURES_PATH)
drought_features["gridmet_id"] = drought_features["gridmet_id"].astype(str)

keep_drought_cols = [
    "gridmet_id",
    "usdm_week",
    "usdm_week_str",
    "usdm_raw",
    "usdm_category",
    "usdm_intensity",
    "is_abnormally_dry_D0_plus",
    "is_drought_D1_plus",
    "is_severe_D2_plus",
    "is_extreme_D3_plus",
    "usdm_missing_reason",
]

merged = base.merge(
    drought_features[[c for c in keep_drought_cols if c in drought_features.columns]],
    on="gridmet_id",
    how="left",
)

merged.to_csv(OUTPUT_PATH, index=False)

print("Base:", base.shape)
print("Drought features:", drought_features.shape)
print("Merged:", merged.shape)
print("Saved:", OUTPUT_PATH)

display(merged[
    [
        "gridmet_id",
        "Name",
        "fire_start_date",
        "usdm_week",
        "usdm_category",
        "usdm_intensity",
        "AcresBurned",
        "severity_tier",
    ]
].head())


## 10. Quality checks

In [ ]:
check = pd.read_csv(OUTPUT_PATH)

print("Rows:", check.shape[0])
print("Missing usdm_intensity:", check["usdm_intensity"].isna().sum())

print("\nUSDM category counts:")
display(check["usdm_category"].value_counts(dropna=False).sort_index())

print("\nUSDM intensity counts:")
display(check["usdm_intensity"].value_counts(dropna=False).sort_index())

if "severity_tier" in check.columns:
    print("\nUSDM intensity by severity tier, normalized by row:")
    display(pd.crosstab(
        check["severity_tier"],
        check["usdm_intensity"],
        normalize="index"
    ).round(3))

if "year" in check.columns:
    print("\nMedian USDM intensity by year:")
    display(check.groupby("year")["usdm_intensity"].median().to_frame("median_usdm_intensity"))


## 11. Final output

If this notebook works, your drought-enriched modeling table is:

```text
data/processed/calfire_with_gridmet_terrain_drought.csv
```

Recommended README wording:

```text
Drought features were sampled at each wildfire ignition coordinate using weekly gridded U.S. Drought Monitor GeoTIFFs. For each incident, the pipeline selected the most recent USDM map available before the fire start date to avoid temporal leakage, then encoded the sampled drought category into D0/D1/D2/D3/D4-derived intensity and indicator features.
```
